## Combining Datasets for all time horizons

This notebook condenses the feature selection, cleanup, and engineered fields from `combine_datasets_reference.ipynb` into one reusable function. Running the final loop will create datasets for the 1, 2, 4, 6, and 12-hour prediction horizons in `combined_data`.

### Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
RAW_DATA_DIR = NOTEBOOK_DIR / "raw_data"
OUTPUT_DIR = NOTEBOOK_DIR / "combined_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = [1, 2, 4, 6, 12]
LOOKBACK_HOURS = 1

### Load the three source datasets

In [2]:
faa_df = pd.read_csv(RAW_DATA_DIR / "FAA_dataset.csv", low_memory=False)
departures_df = pd.read_csv(RAW_DATA_DIR / "SFO_Departures_dataset.csv", low_memory=False)
noaa_df = pd.read_csv(RAW_DATA_DIR / "SFO_NOAA_dataset.csv", low_memory=False)

### Merge departures with FAA aircraft data

Tail numbers are normalized before the merge, and flights without matching FAA information are removed.

In [3]:
departures_df["tail_key"] = (
    departures_df["mod_tail_number"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"^N", "", regex=True)
)
faa_df["tail_key"] = (
    faa_df["N-NUMBER"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"^N", "", regex=True)
)

departures_faa_df = departures_df.merge(
    faa_df,
    on="tail_key",
    how="left",
    validate="many_to_one",
    indicator=True,
)
departures_faa_df.columns = (
    departures_faa_df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\(\)/\-]+", "_", regex=True)
    .str.replace(r"[^\w]", "", regex=True)
    .str.strip("_")
)
departures_faa_cleaned = departures_faa_df.dropna(subset=["n_number"]).copy()
departures_faa_cleaned["scheduled_dep_dt"] = pd.to_datetime(
    departures_faa_cleaned["scheduled_dep_dt"], errors="coerce"
)

### Prepare weather and airport history data

In [4]:
weather_features = [
    "visibility",
    "ceiling_height",
    "wind_speed",
    "wind_direction",
    "temperature",
    "dew_point_temperature",
    "relative_humidity",
    "altimeter",
]

weather_df = noaa_df[["DATE"] + weather_features].rename(columns={"DATE": "weather_source_dt"})
weather_df["weather_observation_dt"] = pd.to_datetime(weather_df["weather_source_dt"], errors="coerce")
weather_df = weather_df.dropna(subset=["weather_observation_dt"]).sort_values("weather_observation_dt")

airport_events = departures_df[["actual_dep_dt", "departure_delay_minutes"]].copy()
airport_events["actual_dep_dt"] = pd.to_datetime(airport_events["actual_dep_dt"], errors="coerce")
airport_events["departure_delay_minutes"] = pd.to_numeric(airport_events["departure_delay_minutes"], errors="coerce")
airport_events = airport_events.dropna(subset=["actual_dep_dt", "departure_delay_minutes"]).sort_values("actual_dep_dt")

### Selecting the model fields

These are the same selected fields used in the reference notebook.

In [5]:
departures_features = [
    "carrier_code",
    "destination_airport",
    "scheduled_elapsed_time_minutes",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "sched_dep_hour",
    "outcome",
    "days_until_holiday",
    "days_from_holiday",
]
faa_features = [
    "year_mfr", 
    "model", 
    "no_seats"
]
metadata_columns = [
    "date", 
    "scheduled_dep_dt", 
    "actual_dep_dt"
]

target_column = "departure_delay_minutes"

selected_columns = (metadata_columns + departures_features + faa_features + weather_features + [target_column])

### Recent airport helper function

For each flight, the helper uses the one-hour window immediately before its prediction cutoff.

In [6]:
def get_recent_airport_delay_stats(row, horizon_hours):
    prediction_cutoff = row["scheduled_dep_dt"] - pd.Timedelta(hours=horizon_hours)
    window_start = prediction_cutoff - pd.Timedelta(hours=LOOKBACK_HOURS)

    recent_departures = airport_events[(airport_events["actual_dep_dt"] >= window_start) & (airport_events["actual_dep_dt"] <= prediction_cutoff)]

    return pd.Series({
        "airport_delay_average_1h": recent_departures["departure_delay_minutes"].mean(),
        "airport_delay_stddev_1h": recent_departures["departure_delay_minutes"].std(),
        "airport_departures_observed_1h": len(recent_departures),
    })

### Build and clean the horizon dataset

In [7]:
def build_combined_dataset(horizon_hours):
    flights = departures_faa_cleaned.copy()
    flights["weather_cutoff_dt"] = (flights["scheduled_dep_dt"] - pd.Timedelta(hours=horizon_hours))

    combined = pd.merge_asof(
        flights.dropna(subset=["weather_cutoff_dt"]).sort_values("weather_cutoff_dt"),
        weather_df,
        left_on="weather_cutoff_dt",
        right_on="weather_observation_dt",
        direction="backward",
        tolerance=pd.Timedelta(minutes=60),
        suffixes=("", "_weather"),
    )

    combined = combined.dropna(subset=["weather_observation_dt"])
    combined = combined[selected_columns].dropna().copy()

    combined["date"] = pd.to_datetime(combined["date"], errors="coerce")
    combined["scheduled_dep_dt"] = pd.to_datetime(combined["scheduled_dep_dt"], errors="coerce")
    combined["actual_dep_dt"] = pd.to_datetime(combined["actual_dep_dt"], errors="coerce")
    combined["year_mfr"] = pd.to_numeric(combined["year_mfr"], errors="coerce")

    combined["aircraft_age"] = combined["year"] - combined["year_mfr"]
    combined.loc[combined["aircraft_age"] < 0, "aircraft_age"] = np.nan
    combined["week_num"] = ((combined["date"].dt.dayofyear - 1) // 7 + 1).astype("int8")
    combined["temperature_dewpoint_spread"] = (combined["temperature"] - combined["dew_point_temperature"])

    # Applying aggregation for each row
    recent_delay_features = combined.apply(
        get_recent_airport_delay_stats,
        axis=1,
        horizon_hours=horizon_hours,
    )

    combined = pd.concat([combined, recent_delay_features], axis=1)

    # Ensuring departure delay matches scheduled departure - actual departure
    combined = combined.dropna(subset=["aircraft_age"]).copy()
    timestamp_delay = (combined["actual_dep_dt"] - combined["scheduled_dep_dt"]).dt.total_seconds().div(60)
    reported_delay = pd.to_numeric(combined[target_column], errors="coerce")
    delay_matches = (
        timestamp_delay.notna()
        & reported_delay.notna()
        & np.isclose(timestamp_delay, reported_delay, atol=0.5)
    )
    combined = combined.loc[delay_matches].copy()

    combined["airport_delay_average_1h"] = combined["airport_delay_average_1h"].fillna(0.0)
    combined["airport_delay_stddev_1h"] = combined["airport_delay_stddev_1h"].fillna(0.0)

    combined = combined.sort_values("scheduled_dep_dt").reset_index(drop=True)
    combined = combined[[column for column in combined.columns if column != target_column] + [target_column]]
    return combined

### Quick check on first horizon

In [8]:
# combined_df = build_combined_dataset(1)
# display(combined_df)

In [9]:
# missing_summary = pd.DataFrame({
#     "column_name": combined_df.columns,
#     "dtype": combined_df.dtypes.astype(str).values,
#     "missing_count": combined_df.isna().sum().values,
#     "missing_percent": (combined_df.isna().mean().values * 100).round(3),
# }).reset_index(drop=True)

# display(missing_summary)

### Create and save all five datasets

Each CSV is written to `combined_data` and includes its prediction horizon in the filename.

In [10]:
for horizon_hours in HORIZONS:
    combined_df = build_combined_dataset(horizon_hours)
    output_path = OUTPUT_DIR / f"combined_data_{horizon_hours}h.csv"
    combined_df.to_csv(output_path, index=False)
    print(
        f"Saved {output_path.name}: "
        f"{combined_df.shape[0]:,} rows x {combined_df.shape[1]} columns"
    )

Saved combined_data_1h.csv: 334,086 rows x 32 columns
Saved combined_data_2h.csv: 335,918 rows x 32 columns
Saved combined_data_4h.csv: 339,996 rows x 32 columns
Saved combined_data_6h.csv: 339,290 rows x 32 columns
Saved combined_data_12h.csv: 346,464 rows x 32 columns
